# 03b. Evaluación de t-SNE, UMAP y HDBSCAN

Este notebook explora técnicas de reducción de dimensionalidad no lineal (t-SNE y UMAP) y evalúa su capacidad para revelar la estructura topológica de los datos, comparando los clústeres originales obtenidos con K-Means y GMM frente a nuevos clústeres basados en densidad usando HDBSCAN sobre el espacio UMAP.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
import sys

# Importar funciones de src
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.clustering import run_tsne_projection, run_umap_projection, run_hdbscan, run_kmeans
from sklearn.preprocessing import StandardScaler
import joblib

# Configurar Plotly para visualizar correctamente
import plotly.io as pio
pio.templates.default = "plotly_white"

## 1. Cargar Datos y Modelos Anteriores

In [2]:
# Cargar dataset procesado con los clústeres de K-Means y GMM
df_path = ROOT_DIR / 'data' / 'processed' / 'sensor_Crop_Dataset_clustered.csv'
df = pd.read_csv(df_path)

print(f'Dataset cargado: {df.shape}')

feature_cols = ['Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Humidity', 'pH_Value', 'Rainfall']
X = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Dataset cargado: (20000, 12)


## 2. Reducción con t-SNE

In [3]:
print("Ejecutando t-SNE...")
X_tsne = run_tsne_projection(X_scaled, perplexity=30.0, random_state=42)

df_proj = df.copy()
df_proj['tsne_1'] = X_tsne[:, 0]
df_proj['tsne_2'] = X_tsne[:, 1]

fig_tsne = px.scatter(
    df_proj, x='tsne_1', y='tsne_2', color=df_proj['kmeans_cluster'].astype(str),
    hover_data=['Crop', 'Soil_Type'],
    title='Proyección t-SNE coloreada por K-Means Cluster (Original)',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_tsne.show()

Ejecutando t-SNE...


c:\Users\darye\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\darye\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\darye\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\darye\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\darye\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

## 3. Reducción con UMAP

In [4]:
print("Ejecutando UMAP...")
X_umap = run_umap_projection(X_scaled, n_neighbors=15, min_dist=0.1, random_state=42)

df_proj['umap_1'] = X_umap[:, 0]
df_proj['umap_2'] = X_umap[:, 1]

fig_umap = px.scatter(
    df_proj, x='umap_1', y='umap_2', color=df_proj['kmeans_cluster'].astype(str),
    hover_data=['Crop', 'Soil_Type'],
    title='Proyección UMAP coloreada por K-Means Cluster (Original)',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_umap.show()

Ejecutando UMAP...


c:\Users\darye\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



## 4. Efecto del Clustering sobre UMAP: K-Means vs HDBSCAN

Aplicaremos de manera separada un algoritmo clásico (K-Means) y uno de densidad (HDBSCAN) sobre los componentes de UMAP para ver el efecto.

In [5]:
print("Ejecutando K-Means sobre UMAP...")
# Usamos el mismo k óptimo encontrado antes (ej. 5)
k_optimal = df_proj['kmeans_cluster'].nunique()
labels_km_umap, _ = run_kmeans(X_umap, k=k_optimal, random_state=42)
df_proj['kmeans_on_umap'] = labels_km_umap

fig_km_umap = px.scatter(
    df_proj, x='umap_1', y='umap_2', color=df_proj['kmeans_on_umap'].astype(str),
    hover_data=['Crop', 'Soil_Type'],
    title=f'K-Means aplicado sobre componentes de UMAP (k={k_optimal})',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_km_umap.show()

Ejecutando K-Means sobre UMAP...


In [6]:
print("Ejecutando HDBSCAN sobre UMAP...")
# Aplicamos HDBSCAN sobre UMAP. min_cluster_size ajusta la granularidad.
labels_hdbscan, hdbscan_model = run_hdbscan(X_umap, min_cluster_size=50)

df_proj['hdbscan_cluster'] = labels_hdbscan

n_clusters = len(set(labels_hdbscan)) - (1 if -1 in labels_hdbscan else 0)
n_noise = list(labels_hdbscan).count(-1)
print(f"Clústeres encontrados por HDBSCAN: {n_clusters}")
print(f"Puntos considerados ruido (outliers, cluster -1): {n_noise}")

# Visualizar
fig_hdb = px.scatter(
    df_proj, x='umap_1', y='umap_2', color=df_proj['hdbscan_cluster'].astype(str),
    hover_data=['Crop', 'Soil_Type'],
    title=f'HDBSCAN aplicado sobre UMAP ({n_clusters} clusters encontrados)',
    opacity=0.7,
    color_discrete_sequence=['lightgray'] + px.colors.qualitative.Alphabet # Ruido en gris
)
fig_hdb.show()

Ejecutando HDBSCAN sobre UMAP...


c:\Users\darye\anaconda3\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning:

The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.



Clústeres encontrados por HDBSCAN: 2
Puntos considerados ruido (outliers, cluster -1): 3577


## 5. Comparativa Cuantitativa

In [11]:
from IPython.utils import importstring
from IPython.core import async_helpers
from IPython import display
from sklearn.metrics import silhouette_score

# Calcular Silueta en el espacio ORIGINAL para ser justos en la comparativa de qué tan coherentes son los grupos
mask = labels_hdbscan != -1

sil_kmeans_orig = silhouette_score(X_scaled, df_proj['kmeans_cluster'])
sil_kmeans_umap = silhouette_score(X_scaled, df_proj['kmeans_on_umap'])

if n_clusters > 1:
    sil_hdbscan = silhouette_score(X_scaled[mask], labels_hdbscan[mask])
else:
    sil_hdbscan = float('nan')

print("=== SILHOUETTE SCORE (Evaluado en espacio original de 7 dimensiones) ===")
print(f"K-Means (Original): {sil_kmeans_orig:.4f}")
print(f"K-Means (Sobre UMAP): {sil_kmeans_umap:.4f}")
print(f"HDBSCAN (Sobre UMAP) [sin ruido]: {sil_hdbscan:.4f}")

# Tabla de contingencia para comparar cómo HDBSCAN agrupó respecto al K-Means original
ct = pd.crosstab(df_proj['kmeans_cluster'], df_proj['hdbscan_cluster'], rownames=['K-Means Orig'], colnames=['HDBSCAN'])
print("=== Contingencia: K-Means Original vs HDBSCAN ===")
print(ct)

=== SILHOUETTE SCORE (Evaluado en espacio original de 7 dimensiones) ===
K-Means (Original): 0.0970
K-Means (Sobre UMAP): 0.0655
HDBSCAN (Sobre UMAP) [sin ruido]: 0.0263
=== Contingencia: K-Means Original vs HDBSCAN ===
HDBSCAN        -1     0   1
K-Means Orig               
0             574  3428  25
1             863  3369  16
2             838  2955  23
3             789  3047  90
4             513  3470   0
